In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from collections import Counter
from itertools import chain
import ast # 문자열로 저장된 리스트를 파이썬 리스트로 안전하게 변환하기 위해 필요
import os

Mounted at /content/drive


In [ ]:
review = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/review_final_merged.csv')
review = review[['Review_UID', 'Raw_Text', 'Cleaned_Text', 'Tokenized_Text']]
review.head()

,Review_UID,Raw_Text,Cleaned_Text,Tokenized_Text
0,A1_S01_R001,로제반반 전화로 주문하여 방문으로 가져간 사람입니다! 너무 맛있게 잘먹었습니다!! ...,로제반반 전화로 주문하여 방문으로 가져간 사람입니다 너무 맛있게 잘먹었습니다 신선한...,"['전화', '주문', '가져가다', '사람', '맛있다', '신선하다', '재료'..."
1,A1_S01_R002,마라엽떡 맛있어요. 밥 비벼먹어도 굿!,마라엽떡 맛있어요 밥 비벼먹어도 굿,"['마라엽떡', '맛있다', '비비다']"
2,A1_S01_R003,너무맛있어요~~~~!!!!!! 직원분들친절하시공ㅎ 자주애용해요ㅎ,너무맛있어요 직원분들친절하시공 자주애용해요,"['맛있다', '직원', '친절하다', '시공', '애용']"
3,A1_S01_R004,2층 홀도좋고 친절하시고 양도많고 너무좋네요~~,2층 홀도좋고 친절하시고 양도많고 너무좋네요,"['친절하다', '양', '많다']"
4,A1_S01_R006,엽닭 존맛입니다,엽닭 존맛입니다,"['엽닭', '존맛']"


In [ ]:
# 🔹 엽떡 리뷰 오타 교정 및 대체어 적용

# 교정 사전 정의
replace_dict = {

    # 🍽 맞춤법 / 철자 오류 교정
    '마싯어요': '맛있어요',
    '마싯다': '맛있다',
    '마싰어요': '맛있어요',
    '맛나요': '맛있어요',
    '맛잇어요': '맛있어요',
    '맛잇어여': '맛있어요',
    '맛잇어용': '맛있어요',
    '맛잇습니당': '맛있습니다',
    '맛잇슴다': '맛있습니다',
    '맛잇었어요': '맛있었어요',
    '맛잇었어용': '맛있었어요',
    '맛잇었습니당': '맛있었습니다',
    '맛잇음': '맛있음',
    '맛앗어요': '맛있어요',
    '맛이떠요': '맛있어요',
    '맛있었오요': '맛있어요',
    '맛있어용': '맛있어요',
    '맛있어여': '맛있어요',
    '맛있어영': '맛있어요',
    '맛있어욬': '맛있어요',
    '맛있겠': '맛있게',
    '맛있습니당': '맛있습니다',
    '맛있습니당~': '맛있습니다',
    '맛있습니당!': '맛있습니다',
    '맛있었습니당': '맛있었습니다',
    '맛있었습니당당': '맛있었습니다',

    # 🧡 비표준어 / 귀여운 어미 교정

    '좋아용': '좋아요',
    '좋아여': '좋아요',
    '좋앙': '좋아요',
    '좋앗어요': '좋았어요',
    '좋앗습니당': '좋았습니다',
    '좋앗습니다': '좋았습니다',
    '좋앗습니당~': '좋았습니다',
    '좋앗슴다': '좋았습니다',
    '좋앗어용': '좋았어요',
    '좋앗어영': '좋았어요',
    '좋앗습니다용': '좋았습니다',
    '좋네여': '좋네요',
    '예용': '이에요',
    '해용': '해요',
    '욬': '요',
    '닼': '다',
    '욤': '요',
    '욥': '요',
    '좋공': '좋고',
    '쵝오': '최고',
    '최공': '최고',
    '입니당': '입니다',
    '꾸르맛탱': '꿀맛이에요',
    '조하요': '좋아요',
    #'좋아유' : '좋아요', <-LDA 이후 발견


    # 💬 강조 / 반복 표현 축약

    '너무너무너무': '너무',
    '너무너무': '너무',
    '넘넘넘': '너무',
    '넘넘': '너무',
    '넘넘넘넘': '너무',
    '느므': '너무',
    '느무': '너무',
    '너모': '너무',
    '넘흐넘흐': '너무',
    '넘맛있': '너무 맛있',
    '넘맛있어요': '너무 맛있어요',
    '넘넘맛있어요': '너무 맛있어요',
    '넘맛잇어요': '너무 맛있어요',
    '넘넘넘맛있어요': '너무 맛있어요',
    '진짜진짜': '정말',
    '최고최고': '최고',
    '짱짱': '짱',
    '쵝오': '최고',
    '쵝오에요': '최고예요',
    '쵝오입니당': '최고입니다',
    '대박이에요': '정말 좋아요',
    '완젼': '완전',
    '졍말': '정말',
    'ㅈㄴ': '정말',
    '좋아오': '좋아요',

    # 😄 친절/불친절 관련 표현

    '친절하세여': '친절해요',
    '친절쓰': '친절해요',
    '불친절쓰': '불친절해요',
    '불친절하세여': '불친절해요',
    '최악이었습니당': '최악이었습니다',


    # 💬 기타 감정/강조 표현

    '감사합니당': '감사합니다',
    '사랑입니당': '사랑입니다',
    '최고입니당': '최고입니다',
    '최고에용': '최고예요',


    # 🙋‍♀️ 맞춤법 오류 / 구어체

    '괜찬아요': '괜찮아요',
    '괜찬았어요': '괜찮았어요',
    '괜찬음': '괜찮음',
    '괜찬았습니당': '괜찮았습니다',
    '드러워요': '더러워요',
    '쥔장': '주인장',
    '괜차늠': '괜찮음',
    'ㄱㅊ': '괜찮',
    '괜찬': '괜찮음',
    'ㅂㄹ': '별로',
    '하시공': '하시고',
    '엽기떡보끼': '엽기떡볶이',
    '꺼같네요': '것 같네요',
    '몬가': '뭔가',
    '떢볶이': '떡볶이',
    '떢': '떡',
    '안닦았는지': '안 닦았는지',
    '더럽도': '더럽고',
    '보이길래': '보이기에',
    '바랬는데': '바랐는데',
    '시캬서': '시켜서',
    '기달려서': '기다려서',
    '기달림': '기다림',
    '기달렸어요': '기다렸어요',
    '기달림니다': '기다립니다',
    '배달포장홀': '배달 포장 홀',
    '불친절힙니다': '불친절합니다',
    '덜매운맛': '덜 매운맛',
    '더럽단': '더럽다는',
    '가섯비': '가성비',
    '핸드폰하기': '휴대폰 하기',
    '보여주심': '보여주셨어요',
    '직원부이': '직원분이',
    '기달렸어요': '기다렸어요',
    '기달림': '기다림',
    '기달려서': '기다려서',
    '안닦았는지': '안 닦았는지',
    '더럽도': '더럽고',
    '보이길래': '보이기에',
    '바랬는데': '바랐는데',
    '시캬서': '시켜서',
    '지저분한적이': '지저분한 적이',
    '배달시켜서': '배달 시켜서',
    '배달시켰는데': '배달 시켰는데',
    '떡볶이국물': '떡볶이 국물',
    '힙니다': '합니다',
    '웠구': '웠고',
    '로재': '로제',
    '해따': '했다',

    # 🍽 맞춤법 / 철자 오류 교정
    '엽똑': '엽떡',
    '마싯게': '맛있게',
    '마싯슴다': '맛있습니다',
    '맛나요': '맛있어요',
    '주뮨': '주문',
    '죠아요': '좋아요',
    '맛잡': '맛집',
    '맛잇음': '맛있음',
    '마라엽떡는': '마라엽떡은',
    '맛잇어': '맛있어',
    '마싯뇌요': '맛있네요',
    '마싯었러여여여': '맛있었어요',
    '마싯었어요': '맛있었어요',
    '착항맛': '착한맛',
    '꾸르맛': '꿀맛',
    '쫀맛': '존맛',
    '존맛탱': '정말 맛있어요',
    '꾸르맛탱': '꿀맛이에요',
    '바삭치즈만두에': '바삭치즈만두를',
    '맛잇어요': '맛있어요',
    '좋타': '좋다',
    '좋앜': '좋았어요',
    '넘 맛': '너무 맛',
    '너무너누': '너무너무',
    '마싯엉': '맛있어요',
    '마라엽떡 기미': '마라엽떡 기대',
    '조아요': '좋아요',
    '좋앗어요': '좋았어요',
    '굳굳': '좋아요',
    '대따': '아주',
    '꾸르맛': '꿀맛',
    '매장에서 맛있게 잘먹었어요': '매장에서 맛있게 잘 먹었어요',
    '매장왓늗데': '매장 왔는데',
    '마라엽떡 넘 맛있어요': '마라엽떡 너무 맛있어요',
    '굿굿 친절하세용': '굿굿 친절하세요',
    '퍽 불친절해요': '매우 불친절해요',
    '야옹야옹야옹야옹좀짜다야옹야옹': '좀 짜요',
    '포장주문 이용했어요': '포장 주문 이용했어요',
    '2인엽떡': '2인 엽떡',
    '주먹 김밥 에': '주먹김밥에',
    '맛있아사': '맛있어서',
    '늘 맛나요': '늘 맛있어요',
    '맛나게': '맛있게',
    '잘먹었습니다': '잘 먹었습니다',
    '넘조하욤': '너무 좋아요',
    '굿굿굿': '아주 좋아요',
    '마라로제로 유부 추가해서': '마라로제에 유부 추가해서',
    '마라로제엽떡이': '마라로제 엽떡이',
    '좋타': '좋다',
    '좋앜': '좋았어요',
    '누무먼하': '너무 많아',
    '허엉어어어또먹구싶다아': '또 먹고 싶다',
    '더 더보기': '',
    '더러웠어요 셀프코너': '더러웠어요. 셀프 코너',
    '싸가지 없음': '불친절함',
    '젤': '제일',
    '험악해지는거': '험악해지는 거',
    '대따 잘 어울리고': '아주 잘 어울리고',
    '떡볶기': '떡볶이',
    '착항맛': '착한맛',
    '로재': '로제',
    '바삭치즈만두랑': '바삭치즈만두와',
    '떡볶이 떡 조합': '떡볶이와 떡의 조합',
    '스슥': '살짝',


    # 🍡 그 외 표현 통일
    '맛잇당': '맛있어요',
    '맛잇네용': '맛있네요',
    '맛잇엉': '맛있어요',
    '맛잇어용용': '맛있어요',
    '맛잇어용용용': '맛있어요',
    '맛잇어요용': '맛있어요',
    '맛잇어요용용': '맛있어요',
    '맛잇어요용용용': '맛있어요',
    '맛잇어요오': '맛있어요',
    '맛잇어용오': '맛있어요',
    '맛잇음당': '맛있어요',
    '맛잇네여': '맛있네요',
    '맛나네여': '맛있네요',
    '맛있었어용': '맛있었어요',
    '맛있었어여': '맛있었어요',
    '먹었오용': '먹었어요',
    '맛남': '맛있음',
    '맛잇어요': '맛있어요',
    '맛잇어여': '맛있어요',
    '맛잇어용': '맛있어요',
    '맛잇습니당': '맛있습니다',
    '맛잇었어요': '맛있었어요',
    '맛잇었어용': '맛있었어요',
    '맛잇었습니당': '맛있었습니다',
    '맛잇어요용': '맛있어요',
    '맛잇어요오': '맛있어요',
    '맛잇어요용용': '맛있어요',
    '맛잇어요용용용': '맛있어요',
    '맛잇엉': '맛있어요',
    '맛잇당': '맛있어요',
    '맛잇네여': '맛있네요',
    '맛잇네용': '맛있네요',
    '맛잇습니당당': '맛있습니다',
    '맛잇슴다': '맛있습니다',
    '맛잇어용용': '맛있어요',
    '맛잇어용용용': '맛있어요',
    '맛잇어요용용용': '맛있어요',
    '맛잇어용오': '맛있어요',
    '맛있었어용': '맛있었어요',
    '맛 있어요': '맛있어요',
    '좋아용': '좋아요',
    '좋아여': '좋아요',
    '좋앙': '좋아요',
    '좋앗어요': '좋았어요',
    '좋앗습니다': '좋았습니다',
    '좋앗습니당': '좋았습니다',
    '좋앗습니당~': '좋았습니다',
    '좋앗슴다': '좋았습니다',
    '좋앗어용': '좋았어요',
    '좋앗어영': '좋았어요',
    '좋앗습니다용': '좋았습니다',
    '좋네여': '좋네요',
    '좋아요용': '좋아요',
    '좋아용용': '좋아요',
    '좋아용용용': '좋아요',
    '굿굿굿': '좋아요',
    '굿굿굿굿': '좋아요',
    '짱맛탱이에요': '정말 맛있어요',
    '존맛탱이에요': '정말 맛있어요',
    '굿좋아요': '좋아요',
    '굳굳좋아요': '좋아요',
    '넘맛있어요': '너무 맛있어요',
    '넘넘맛있어요': '너무 맛있어요',


     # 🍲 표현 및 띄어쓰기 교정
    '맛있게잘먹었습니다': '맛있게 잘 먹었습니다',
    '맛있게 잘먹었어요': '맛있게 잘 먹었어요',
    '맛있게잘먹었어요': '맛있게 잘 먹었어요',
    '맛있게잘먹었습니당': '맛있게 잘 먹었습니다',
    '맛있게잘먹었습니당': '맛있게 잘 먹었습니다',
    '맛있게 잘먹었습니당': '맛있게 잘 먹었습니다',
    '맛있게잘먹었습니당': '맛있게 잘 먹었습니다',
    '맛있게잘먹었습니당!': '맛있게 잘 먹었습니다',
    '맛있게 잘목었습니다': '맛있게 잘 먹었습니다',
    '맛있게잘목었어요': '맛있게 잘 먹었어요',
    '잘묵엇어요': '잘 먹었어요',
    '잘묵엇습니더': '잘 먹었습니다',
    '잘묵엇숩니더': '잘 먹었습니다',
    '맛있게묵엇습니더': '맛있게 먹었습니다',
    '잘묵었어요': '잘 먹었어요',
    '잘먹었습니당': '잘 먹었습니다',
    '잘먹었어요용': '잘 먹었어요',
    '맛있게잘묵었어요': '맛있게 잘 먹었어요',
    '맛있게잘묵었습니당': '맛있게 잘 먹었습니다',
    '잘묵었습니당': '잘 먹었습니다',
    '잘묵었어용': '잘 먹었어요',
    '잘먹었어요': '잘 먹었어요',
    '잘먹었습니당': '잘 먹었습니다',
    '맛잇': '맛있',


    # 🧂 발음 및 비표준어
    '엽떡머는날': '엽떡 먹는 날',
    '엽떡머는날': '엽떡 먹는 날',
    '엽떡머는날': '엽떡 먹는 날',
    '엽떡마싯다': '엽떡 맛있다',
    '마라엽떡너무맛있어요': '마라엽떡 너무 맛있어요',
    '마라엽떡최고시당': '마라엽떡 최고입니다',
    '마라엽떡최고입니당': '마라엽떡 최고입니다',
    '엽떡너무좋아용': '엽떡 너무 좋아요',
    '엽떡너무좋아요': '엽떡 너무 좋아요',
    '엽떡너무맛있어용': '엽떡 너무 맛있어요',
    '엽떡너무맛있어요': '엽떡 너무 맛있어요',
    '엽떡맛잇어요': '엽떡 맛있어요',
    '엽떡맛있어용': '엽떡 맛있어요',
    '엽떡맛있습니당': '엽떡 맛있습니다',
    '엽떡너무맛있습니당': '엽떡 너무 맛있습니다',
    '엽떡너무맛있습니당!': '엽떡 너무 맛있습니다',
    '엽떡맛있습니당': '엽떡 맛있습니다',
    '엽떡너무좋아여': '엽떡 너무 좋아요',
    '엽떡너무좋아용': '엽떡 너무 좋아요',



    # 🍜 맞춤법 / 철자 오류 교정
    '맛있게 잘목었습니다': '맛있게 잘 먹었습니다',
    '넘 맛 있어여': '너무 맛있어요',
    '오눌따라': '오늘따라',
    '맛있었어용': '맛있었어요',
    '맛있어용': '맛있어요',
    '맛있었어용': '맛있었어요',
    '너모 맛있어요': '너무 맛있어요',
    '떢볶기': '떡볶이',
    '맛있 마라엽떡': '맛있는 마라엽떡',
    '떡복기': '떡볶이',
    '떡복이는': '떡볶이는',
    '맛있게 잘하세요': '맛있게 잘 하세요',
    '좋아하는 엽떡에': '좋아하는 엽떡은',
    '매번 잘 먹고 있어오': '매번 잘 먹고 있어요',
    '떡볶기 먹는날': '떡볶이 먹는 날',
    '엽떡머는날': '엽떡 먹는 날',
    '너뮤': '너무',
    '좋을것 같아요': '좋을 것 같아요',
    '마라엽떡 공덕점 최고입니당': '마라엽떡 공덕점 최고입니다',
    '마라엽떡 최고시당': '마라엽떡 최고입니다',
    '마라엽떡 중독입니당': '마라엽떡 중독입니다',
    '마포공덕점이 최고 맛나용': '마포공덕점이 최고 맛나요',
    '맛있눈고야': '맛있는 거야',
    '너무맛있었어요': '너무 맛있었어요',
    '맛있잇어요': '맛있어요',
    '마라로제떡볶이 처음 먹어보눈데': '마라로제떡볶이 처음 먹어보는데',
    '마딛': '맛있다',
    '역쉬': '역시',
    '맛이변했어요': '맛이 변했어요',
    '좋이않았어요': '좋지 않았어요',
    '서비스엉망이에요': '서비스 엉망이에요',
    '배달비 사처넌': '배달비 사천원',
    '매운 움식에': '매운 음식에',
    '엽떡오리지널인데': '엽떡 오리지널인데',
    '다륻 지점에': '다른 지점에',
    '최구': '최고',
    '꿀꿀할때': '꿀꿀할 때',
    '최고예요': '최고예요',
    '마싯다': '맛있다',
    '불손한태도로': '불손한 태도로',
    '먹고싶다': '먹고 싶다',
    '먹고싶어요': '먹고 싶어요',
    '싸가지없음': '불친절함',
    '기분더러웠는데': '기분 더러웠는데',
    '짧은 남자직원교육좀': '짧은 남자 직원 교육 좀',
    '불손한태도로': '불손한 태도로',
    '엽떡앱에는': '엽떡 앱에는',
    '엽떡이아니라무슨': '엽떡이 아니라 무슨',
    '포장주문하는데': '포장 주문하는데',
    '배달시켜목는': '배달 시켜먹는',
    '양배추 파 등': '양배추, 파 등',
    '맛있어요 다만': '맛있어요. 다만',
    '마싯고': '맛있고',
    '엽떡 존맛탱쓰': '엽떡 정말 맛있어요',
    '못먹엇어오': '못 먹었어요',
    '아숩아숩': '아쉽아쉽',
    '너무너무 더움': '너무너무 더워요',
    '더워서 헥헥대면서도': '더워서 헉헉대면서도',
    '에어컨좀': '에어컨 좀',
    '설치하시는게어쩔지': '설치하시는 게 어떨지',
    '주먹김밥 완전 질어서 떡밥이고': '주먹김밥이 완전 질어서 떡밥 같고',
    '엽떡 여의도에서 먹을수 있는 자리가 있어요': '엽떡 여의도에서 먹을 수 있는 자리가 있어요',
    '언재먹어도': '언제 먹어도',
    '젤 맛있는 곳': '제일 맛있는 곳',
    '마라맛 너무 맛있어요': '마라 맛 너무 맛있어요',
    '조아요': '좋아요',
    '좋타': '좋다',
    '좋앗어요': '좋았어요',
    '엽떡머는날': '엽떡 먹는 날',
    '너무 맛있습니당': '너무 맛있습니다',
    '최고입니당': '최고입니다',
    '갠춘': '괜찮',
    '맛있뜌': '맛있다',
    '야징':'야지',
    '듈이' : '둘이',
    '좋습니댜' : '좋습니다',
    '계란찌' : '계란찜',
    '안되서' : '안돼서',
    '최고에여' : '최고예요',
    '먀장' : '매장',
    '되서': '돼서',


    # 🍜 맞춤법/철자 오류
    '을매나': '얼마나',
    '잇어요': '있어요',
    '잇습니당': '있습니다',
    '잇었어요': '있었어요',
    '맛잇어요': '맛있어요',
    '맛잇었어요': '맛있었어요',
    '맛잇는디': '맛있는데',
    '맛잇음': '맛있음',
    '맛잇습니당': '맛있습니다',
    '맛잇게': '맛있게',
    '맛잇어용': '맛있어요',
    '맛잇엇어요': '맛있었어요',
    '맛잇엇습니당': '맛있었습니다',
    '맛잇어여': '맛있어요',
    '맛잇었어용': '맛있었어요',
    '맛잇어용': '맛있어요',
    '마싯어요': '맛있어요',
    '마싯었어요': '맛있었어요',
    '마싯습니당': '맛있습니다',
    '마싯다': '맛있다',


    # 😊 구어체 / 음성어
    '맵찔이': '매운 음식을 잘 못 먹는 사람',
    '맵찔이라': '매운 음식을 잘 못 먹어서',
    '굳굳': '좋아요',
    '굿굿': '좋아요',
    '굿': '좋아요',
    '조아요': '좋아요',
    '좋아용': '좋아요',
    '좋앗어요': '좋았어요',
    '좋앗습니당': '좋았습니다',
    '좋앗어요용': '좋았어요',
    '오옹': '오',
    '오홍': '오',
    '오홍홍': '오',
    '조하요': '좋아요',
    '굳굳굳': '좋아요',
    '너무너무': '너무',
    '넘': '너무',
    '너무너무너무': '너무',
    '너무너무너무너무': '너무',
    '굳굳이에요': '좋아요',
    '굳이에요': '좋아요',
    '굿이에요': '좋아요',
    '조하용': '좋아요',

    # 😋 표현 교정
    '잘먹엇어요': '잘 먹었어요',
    '잘먹엇습니당': '잘 먹었습니다',
    '잘묵엇어요': '잘 먹었어요',
    '잘묵엇습니당': '잘 먹었습니다',
    '잘먹었습니당': '잘 먹었습니다',
    '잘먹었어용': '잘 먹었어요',
    '잘먹었어요용': '잘 먹었어요',
    '잘먹었어욤': '잘 먹었어요',
    '잘먹엇어요': '잘 먹었어요',
    '잘먹엇습니당': '잘 먹었습니다',
    '잘묵었어요': '잘 먹었어요',
    '잘묵었습니당': '잘 먹었습니다',
    '잘먹었어용': '잘 먹었어요',

    # 🧂 기타 맞춤법
    '칭구들': '친구들',
    '징짜': '진짜',
    '쫀맛': '존맛',
    '꾸르맛': '꿀맛',
    '꾸르맛탱': '꿀맛이에요',
    '짱맛': '정말 맛있어요',
    '맛나용': '맛나요',
    '맛나욤': '맛나요',
    '맛나여': '맛나요',
    '맛잇당': '맛있다',
    '맛잇당!': '맛있다',
    '맛있습니당': '맛있습니다',
    '맛있었습니당': '맛있었습니다',
    '입니당': '입니다',
    '입니당~': '입니다',
    '입니당!': '입니다',
    '입니당요': '입니다',
    '이죵': '이죠',
    '이쥬': '이죠',
    '번창하세용': '번창하세요',
    '번창하세염': '번창하세요',
    '번창하세욤': '번창하세요',
    '번창하세용~': '번창하세요',
    '번창하세용!!': '번창하세요',
    '넘좋아요': '너무 좋아요',
    '넘좋았어요': '너무 좋았어요',
    '굿굿굿이에요': '좋아요',
    '사랑해용': '사랑해요',
    '사랑입니당': '사랑입니다',

    # 🧾 띄어쓰기
    '잘먹었어요': '잘 먹었어요',
    '잘먹었습니다': '잘 먹었습니다',
    '잘먹었습니당': '잘 먹었습니다',
    '감사합니당': '감사합니다',
    '감사합니당~': '감사합니다',
    '감사합니당!': '감사합니다',
    '고맙습니당': '고맙습니다',
    '사랑합니당': '사랑합니다',
    '맛잇었습니당': '맛있었습니다',
    '맛잇었어용': '맛있었어요',
    '맛잇어용': '맛있어요',


# ✅ 교정 사전 정의
    # 오탈자/은어
    '젤루': '제일',
    '짜용': '짜요',
    '맛있우요': '맛있어요',
    '입니데이': '입니다',
    '넘넘': '너무',
    '넘': '너무',
    '굳굳': '좋아요',
    '굿굿': '좋아요',
    '굿': '좋아요',
    '쫀맛': '정말 맛있어요',
    '꾸르맛': '꿀맛',
    '꿀맛탱': '꿀맛이에요',
    '오옹': '오',
    '오홍': '오',
    '좋아용': '좋아요',
    '맛잇어요': '맛있어요',
    '맛잇었어요': '맛있었어요',
    '맛잇는디': '맛있는데',
    '맛잇엇어요': '맛있었어요',
    '맛잇게': '맛있게',
    '맛잇어용': '맛있어요',
    '맛잇어여': '맛있어요',
    '맛잇습니당': '맛있습니다',
    '맛잇당': '맛있다',
    '맛잇었어용': '맛있었어요',
    '맛잇엇습니당': '맛있었습니다',
    '맛잇어욤': '맛있어요',
    '맛잇엇어용': '맛있었어요',
    '맛잇게요': '맛있어요',
    '맛잇음': '맛있음',
    '마싯어요': '맛있어요',
    '마싯었어요': '맛있었어요',
    '마싯당': '맛있다',
    '입니당': '입니다',
    '입니당요': '입니다',
    '입니당~': '입니다',
    '입니당!': '입니다',
    '감사합니당': '감사합니다',
    '고맙습니당': '고맙습니다',
    '사랑해용': '사랑해요',
    '사랑합니당': '사랑합니다',
    '번창하세용': '번창하세요',
    '번창하세욤': '번창하세요',
    '번창하세염': '번창하세요',
    '맛잇습니당': '맛있습니다',
    '맛잇었습니당': '맛있었습니다',
    '맛잇었어용': '맛있었어요',
    '맛잇어용': '맛있어요',
    '굳이에요': '좋아요',
    '굳굳이에요': '좋아요',
    '엽떡 입니데이': '엽떡입니다',
    '맵찔이': '매운 음식을 잘 못 먹는 사람',
    '머리카락 나왔어요 하': '머리카락이 나왔어요. 하',
    '전에안그랬는데독특한향기가나네요': '전에 안 그랬는데 독특한 향기가 나네요',
    '역시엽떡': '역시 엽떡',
    '진심개별로임': '진짜 별로예요',
    '엽기떢볶기': '엽기떡볶이',
    '오리지날': '오리지널',
    '떡볶기가': '떡볶이가',
    '맛있어 요': '맛있어요',
    '맛있겠 잘 먹었습니다': '맛있게 잘 먹었습니다',
    '떡볶이는 엽기떢볶기가': '떡볶이는 엽기떡볶이가',
    '엽떡 입니데이': '엽떡입니다',
    '맵다요': '맵네요',
    '젤루 짜요': '제일 짜요',
    '맛있우요': '맛있어요',
    '맵찔이에겐': '매운 음식을 잘 못 먹는 사람에게는',
    '넘넘맛있어요': '너무 맛있어요',
    '덜매운맛시켰는데 오 매워요': '덜 매운 맛 시켰는데 오 매워요',


   '했네용': '했네요',
    '어용': '어요',
    '맛나요': '맛있어요',
    '짯던거': '짰던 것',
    '짯던거같아요': '짰던 것 같아요',
    '먹었습니더': '먹었습니다',
    '했습니더': '했습니다',
    '맛잇어요': '맛있어요',
    '맛잇엇어요': '맛있었어요',
    '맛잇었어요': '맛있었어요',
    '맛잇었습니당': '맛있었습니다',
    '맛잇게': '맛있게',
    '맛잇당': '맛있다',
    '입니당': '입니다',
    '입니당요': '입니다',
    '입니당~': '입니다',
    '입니당!': '입니다',
    '입니당^^': '입니다',
    '맛있습미당': '맛있습니다',
    '맛잇습니당': '맛있습니다',
    '맛잇었습니당': '맛있었습니다',
    '맛있습니당': '맛있습니다',
    '맛잇어용': '맛있어요',
    '맛잇어여': '맛있어요',
    '맛잇엇어용': '맛있었어요',
    '맛잇었어용': '맛있었어요',
    '맛잇엇어여': '맛있었어요',
    '맛잇어욤': '맛있어요',
    '맛잇어요': '맛있어요',
    '마싯어요': '맛있어요',
    '마싯었어요': '맛있었어요',
    '마싯었습니당': '맛있었습니다',
    '굿굿': '좋아요',
    '굳굳': '좋아요',
    '굿이에요': '좋아요',
    '굳이에요': '좋아요',
    '맛잇는디': '맛있는데',
    '맛잇는대': '맛있는데',
    '했습니당': '했습니다',
    '고맙습니당': '고맙습니다',
    '감사합니당': '감사합니다',
    '마세용': '마세요',
    '하세용': '하세요',
    '하세욤': '하세요',
    '하세염': '하세요',
    '되세용': '되세요',
    '하세여': '하세요',
    '사랑해용': '사랑해요',
    '사랑합니당': '사랑합니다',
    '짱이에용': '짱이에요',
    '마라엽떡 love': '마라엽떡 최고',
    '좋아용': '좋아요',
    '좋아욤': '좋아요',
    '좋아염': '좋아요',
    '좋앙': '좋아요',
    '싫엉': '싫어요',
    '엽떡 입니데이': '엽떡입니다',
    '엽떡 입니데': '엽떡입니다',
    '엽떡 입니데요': '엽떡입니다',
    '했습니데이': '했습니다',
    '있어용': '있어요',
    '없어용': '없어요',
    '맞아용': '맞아요',
    '대박이에용': '대박이에요',
    '대박이네용': '대박이네요',
    '했어용': '했어요',
    '했어욤': '했어요',
    '했어염': '했어요',
    '먹었어용': '먹었어요',
    '먹었어욤': '먹었어요',
    '먹었어염': '먹었어요',
    '먹었습니당': '먹었습니다',
    '봤어용': '봤어요',
    '봤어염': '봤어요',
    '봤어욤': '봤어요',
    '있습니당': '있습니다',
    '했습니데이': '했습니다',
    '좋습니당': '좋습니다',
    '합니다용': '합니다',
    '있습니데이': '있습니다',
    '했습니데': '했습니다',
    '했습니다용': '했습니다',
    '매워용': '매워요',
    '맵네용': '맵네요',
    '괜찮네용': '괜찮네요',
    '괜찮아용': '괜찮아요',
    '괜찮습니당': '괜찮습니다',
    '괜찮았습니당': '괜찮았습니다',
    '좋았습니당': '좋았습니다',
    '했습니디': '했습니다',
    '좋았어용': '좋았어요',
    '좋았어욤': '좋았어요',
    '좋았어염': '좋았어요',
    '괜찮았어용': '괜찮았어요',
    '괜찮았어욤': '괜찮았어요',
    '괜찮았어염': '괜찮았어요',
    '짯어요': '짰어요',
    '짯네용': '짰네요',
    '짯어용': '짰어요',
    '짯어욤': '짰어요',
    '짯어염': '짰어요',
    '묵은쌀로': '묵은 쌀로',
    '지은밥이었고': '지은 밥이었고',
    '짯던거': '짰던 것',
    '짯던거같아요': '짰던 것 같아요',
    '좋았습니더': '좋았습니다',
    '맛있습니더': '맛있습니다',
    '좋았습미당': '좋았습니다',
    '아쉬운거는': '아쉬운 것은',
    '좋았어용': '좋았어요',
    '괜찮은데용': '괜찮은데요',
    '괜찮은데욤': '괜찮은데요',
    '괜찮은데염': '괜찮은데요',
    '같네용': '같네요',
    '같네욤': '같네요',
    '같네염': '같네요',
    '같아요용': '같아요',
    '같아요욤': '같아요',
    '같아요염': '같아요',
    '먹었어염': '먹었어요',
    '배불러용': '배불러요',
    '배불렀어용': '배불렀어요',
    '배불렀어욤': '배불렀어요',
    '배불렀어염': '배불렀어요',
    '먹었음돠': '먹었습니다',
    '맛있었습니당': '맛있었습니다',
    '맛있었습미당': '맛있었습니다',
    '좋았어요용': '좋았어요',
    '괜찮았어요용': '괜찮았어요',
    '괜찮았어요욤': '괜찮았어요',
    '좋았어요욤': '좋았어요',
    '좋았어요염': '좋았어요',
    '좋았어요용': '좋았어요',
    '맛있었어욤': '맛있었어요',
    '맛있었어염': '맛있었어요',
    '좋았습니당': '좋았습니다',
    '묵은쌀로 지은밥이었고 ': '묵은 쌀로 지은 밥이었고 ',

    '맛있어오': '맛있어요',
    '떢볶이': '떡볶이',
    '떡복이': '떡볶이',
    '너모': '너무',
    '맛잇당': '맛있다',
    '맛도리': '맛돌이',
    '배달로드세요': '배달로 드세요',
    '너어어어무': '너무',
    '매워요요': '매워요',
    '안좋은': '안 좋은',
    '않좋은': '안 좋은',
    '맛있구': '맛있고',
    '맛있구요': '맛있고요',
    '않좋아요': '안 좋아요',
    '그럴수있어요': '그럴 수 있어요',
    '그럴수있죠': '그럴 수 있죠',
    '그래슴': '그랬음',
    '너무너무너무너무너무': '너무너무',
    '너무너무너무': '너무너무',
    '너무너무': '너무',
    '지저분해요숟가락에는': '지저분해요. 숟가락에는',
    '잘먹었습니다': '잘 먹었습니다',
    '좋아요요': '좋아요',
    '굳': '굿',
    '죤맛': '존맛',
    '맛잇어요': '맛있어요',
    '좋았습니당': '좋았습니다',
    '맛있습니당': '맛있습니다',
    '맛있당': '맛있다',
    '맛있엉': '맛있어',
    '맛있어염': '맛있어요',
    '맛있어용': '맛있어요',
    '맛잇어용': '맛있어요',
    '맛잇어여': '맛있어요',
    '맛잇어욤': '맛있어요',
    '맛잇어염': '맛있어요',
    '좋앗어요': '좋았어요',
    '좋앗습니당': '좋았습니다',
    '너무너무맛있어요': '너무 맛있어요',
    '존맛탱': '존맛',
    '맛잇습니당': '맛있습니다',
    '맛잇었어요': '맛있었어요',
    '맛잇었습니당': '맛있었습니다',
    '너무너무맛있네요': '너무 맛있네요',
    '맛잇당': '맛있다',
    '너무너무맛있어요': '너무 맛있어요',
    '너무너무맛있네요': '너무 맛있네요',
    '너무너무너무너무': '너무너무',
    '맛있었러욪': '맛있었어요',
    '맛있었으나': '맛있었지만',
    '맛있어여': '맛있어요',
    '맛있어욤': '맛있어요',
    '맛있어염': '맛있어요',
    '맛있었습니당': '맛있었습니다',
    '맛잇어용': '맛있어요',
    '맛잇었어용': '맛있었어요',
    '맛잇었어욤': '맛있었어요',
    '맛잇었어염': '맛있었어요',
    '좋았습니더': '좋았습니다',
    '좋았습미당': '좋았습니다',
    '좋앗어용': '좋았어요',
    '좋앗어욤': '좋았어요',
    '좋앗어염': '좋았어요',
    '매워염': '매워요',
    '매워욤': '매워요',
    '매워여': '매워요',
    '맛잇어요': '맛있어요',
    '맛잇었어요': '맛있었어요',
    '맛잇었습니당': '맛있었습니다',
    '맛잇습니당': '맛있습니다',
    '맛잇었습니당': '맛있었습니다',
    '맛잇어용': '맛있어요',
    '맛잇어여': '맛있어요',
    '맛잇어욤': '맛있어요',
    '맛잇어염': '맛있어요',
    '같아여': '같아요',
    '같아염': '같아요',
    '같아욤': '같아요',
    '좋아욤': '좋아요',
    '좋아염': '좋아요',
    '좋앗어용': '좋았어요',
    '좋앗어욤': '좋았어요',
    '좋앗어염': '좋았어요',
    '맛잇었어용': '맛있었어요',
    '맛잇었어욤': '맛있었어요',
    '맛잇었어염': '맛있었어요',
    '맛있습미당': '맛있습니다',
    '굿굿': '굿',
    '굳굳': '굿',
    '맛있어오': '맛있어요',
    '맛잇엇어요': '맛있었어요',
    '맛잇었어용': '맛있었어요',
    '맛잇었어욤': '맛있었어요',
    '맛잇었어염': '맛있었어요',
    '좋았습니당': '좋았습니다',

    '자시': '잠시',
    '졸라': '',  # 비속어이므로 삭제 또는 문맥에 따라 생략 권장
    '퐁당치즈민두': '퐁당치즈만두',
    '엽기 떡볶이 를': '엽기떡볶이를',
    '없슴': '없음',
    '있슴': '있음',
    '지역여행중일때': '지역 여행 중일 때',
    '아뭏든': '아무튼',
    '잇엇을거에요': '있었을 거예요',
    '써비스': '서비스',
    '로제떡복기': '로제떡볶이',
    '다먹었넹': '다 먹었네요',
    '떡볶이하나먹는데40분기다렸음': '떡볶이 하나 먹는데 40분 기다렸음',
    '주문누락이라는데 말이라도미리해주지 계속기다리라는건 무슨소리임 다시는여기안올듯': '주문 누락이라는데 말이라도 미리 해주지, 계속 기다리라는 건 무슨 소리입니까. 다시는 여기 안 올 듯.',
    '4가지 없고': '싸가지 없고',
    '더럽도': '더럽고',
    '더럽게할것같은': '더럽게 할 것 같은',
    '그랫슨': '그랬음',
    '맛있둠': '맛있음',
    '김밥은국룰이죠': '김밥은 국룰이죠',

    '시캬서': '시켜서',
    '엽기닭도리탕': '엽기 닭도리탕',
    '받질않으시네요': '받질 않으시네요',
    '받질않고': '받질 않고',
    '피한다고다가아니잖아요': '피한다고 다가 아니잖아요',
    '받앗는데': '받았는데',
    '뿔어서': '불어서',
    '이걸 먹으라고요': '이걸 먹으라는 건가요',
    '돈 아깝네요': '돈이 아깝네요',
    '맛잇어용': '맛있어요',
    '좋아용': '좋아요',
    '길가여서': '길가에 있어서',
    '들어옴': '들어옵니다',
    '단무지500원 으로바꾸는데': '단무지 500원으로 바꾸는데',
    '불친절해서': '불친절해서',
    '여알바분': '여자 알바분',
    '넘 귀엽고': '너무 귀엽고',
    '최공': '최고',
    '왕맛탱': '왕맛',
    '오늘은유독짠듯해요': '오늘은 유독 짠 듯해요',
    '그래도물좀넣고': '그래도 물 좀 넣고',
    '좋았어요': '좋았어요',
    '향신료때료부으셧나요': '향신료 때려 부으셨나요',
    '씹혀서': '씹혀서',
    '불어서': '불어서',
    '읭했는데': '의아했는데',
    '마라떡볶이은근': '마라떡볶이가 은근',
    '가섯비': '가성비',
    '초보 맛있었요': '초보 맛있었어요',
    '맛잇어요': '맛있어요',
    '마라로제엽떡': '마라 로제 엽떡',
    '2만원후반대로': '2만원 후반대로',
    '진철합니다': '친절합니다',
    '얏 샤탸': '',
    '도긴개긴': '도긴개긴이에요',
    '기달려서': '기다려서',
    '엽떠': '엽떡',
    '맛있어용': '맛있어요',
    '매워요 쵝오시다': '매워요 최고입니다',
    '싸가지x': '싸가지 없어요',
    '남알바': '남자 알바',
    '11 26 오후 5 55분 쯤': '11월 26일 오후 5시 55분쯤',
    '직원부이': '직원분이',
    '테이블고 닦지': '테이블도 닦지',
    '불친절힙니다': '불친절합니다',
    '불친절하십니다': '불친절합니다',
    '치즈올리는거': '치즈 올리는 거',
    '사장님불러서': '사장님 불러서',
    '일한지 3일차인데': '일한 지 3일 차인데',
    '예쁘게 못올린걸말한거': '예쁘게 못 올린 걸 말한 거',
    '여성알바생': '여자 알바생',
    '헹주로': '행주로',
    '불친절함이 치가떨릴정도': '불친절함이 치가 떨릴 정도',
    '엽떡이쥬': '엽떡이죠',
    '넘넘': '너무너무',
    '날려지네여': '날려지네요',
    '엽떡앱': '엽떡 앱',
    '아는척도': '아는 척도',
    '무안하게': '민망하게',
    '귀찮아해서': '귀찮아서',

    # 여진님 파트 #

'맛있어요이': '맛있어요',
'마라로제엽떡': '마라로제 엽떡',
'무뼈닭발': '무뼈 닭발',
'때': ' 때 ',
'땐': ' 때는 ',
'떡추가': '떡 추가',
'함미다': '합니다',
'가세용': '가세요',
'맛있엉': '맛있어요',
'또': ' 또 ',
'오랫만에': '오랜만에',
'마니': ' 많이 ',
'이예요': '이에요',
'강츄': '강추',
'망있음': '맛있음',
'잘드시네여': '잘 드시네요',
'달걀찜': '계란찜',
'화장실세면데는': '화장실 세면대는',
'음식물찌거기로': '음식물 찌꺼기로',
'파.바옆건물이엿던': '파리바게트 옆 건물이었던',
'나왓던': '나왔던',
'그랫고': '그랬고',
'더요청햇는데': '더 요청했는데',
'나왓는데도': '나왔는데도',
'죄송하단': '죄송하다는',
'잇는': '있는',
'알바생이래도': '알바생이라도',
'테이블뿐이엿던걸로': '테이블뿐이었던 걸로',
'제일쩔엇다': '제일 좋았다',
'괜찮아용': '괜찮아요',
'매꼽하고': '매콤하고',
'서비으로': '서비스로',
'맛있틈': '맛있음',
'라구요': '라고요',
'햇는데': '했는데',
'꿀맛탱': '꿀맛',
'청경하지는': '청결하지는',
'아나라': '아니라',
'응대해주시더라구요': '응대해 주시더라고요',
'맛나당': '맛나다',
'맛났어요': '맛있었어요',
'매웡': '매워요',
'턱봌이는': '떡볶이는',
'쫌': '좀',
'왠만한': '웬만한',
'돌맹이': '돌멩이',
'줄일라고': '줄이려고',
'찍엇어요': '찍었어요',
'배딜': '배달',
'마싯따': '맛있다',
'주방이모': '주방 이모',
'4가지가': '싸가지가',
'하네용': '하네요',
'사각형모양이예요': '사각형 모양이에요',
'조와요': '좋아요',
'는대': '는데',
'맛나여': '맛나요',
'알반지': '알바인지',
'하셔야져': '하셔야죠',
'죄송한다는': '죄송하다는',
'그대론데': '그대로인데',
'조져보았습니다': '먹어보았습니다',
'먹구': '먹고',
'칠려고': '치려고',
'xx싸가지': '싸가지',
'모아져잇는': '모여 있는',
'덩어리가 답니다': '덩어리가 다입니다',
'모양': '모양이',
'맛있써요': '맛있어요',
'걸릴거': '걸릴 것',
'렌즈에': '전자레인지에',
'타지점': '타 지점',
'뛰어다': '뛰어다녔어요',
'맛난거같아용': '맛있는 것 같아요',
'체고': '최고',
'딴지점': '다른 지점',
'먹었네용': '먹었네요',
'멋지신거': '멋지신 것',
'맛대가리가': '맛대가리가',
'무슨 이유인간에': '무슨 이유든지 간에',
'차당하는': '차단하는',
'Long': '긴',
'떡복이': '떡볶이',
'먹어봣는데': '먹어봤는데',
'매어요': '매워요',
'나올려면': '나오려면',
'풀어터져있고': '불어터져 있고',
'베어있지': '배어있지',
'프랜차이점': '프랜차이즈점',
'간도안되어': '간도 안 되어',
'걍': '그냥',
'맛있어요구리': '맛있어요',
'놀랬어요': '놀랐어요',
'최악이에여': '최악이에요',
'같더라두요': '같더라고요',
'가려구요': '가려고요',
'쎄 코': '세스코',
'시겨서': '시켜서',
'습니당': '습니다',
'아용하겠습니다': '이용하겠습니다',
'느껴졌음요': '느껴졌어요',
'잇네용': '있네요',
'왔어용': '왔어요',
'역싀': '역시',
'프랜챠이즈': '프랜차이즈',
'지존답다고나': '지존답다고나',
'담번엔': '다음번엔',
'맛이고자시고': '맛이고 자시고',
'때오라고': '떼오라고',
'땟떠니': '떼었더니',
'죄송요': '죄송',
'양이': ' 양이 ',
'가태요': '같아요',
'담날': '다음날',
'질까바': '질까 봐',
'안되요': '안 돼요',
'후라이팬': '프라이팬',
'볼껄': '볼걸',
'존마탱': '존맛탱',
'갸': '개',
'려구요': '려고요',
'먹었어여': '먹었어요',
'맛있음다': '맛있습니다',
'이엇어요': '이었어요',
'씨유': 'CU',
'엽닭': '엽기닭발',
'떡볶 국': '떡볶이 국',
'시켰는대': '시켰는데',
'되요': '돼요',
'돌이 대화하느라': '둘이 대화하느라',
'잡침요': '잡쳤어요',
'넘나': '너무',
'미쳤어용': '미쳤어요',
'마나서': '많아서',
'떡봌이': '떡볶이',
'맛잇': '맛있',
'타점': '타 지점',
'시켯': '시켰',
'컴플': '컴플레인',
'셧는데': '셨는데',
'봉다리': '봉지',
'둥굴게': '둥글게',
'울집': '우리 집',
'셧음': '셨음',
'됩니당': '됩니다',
'보네용': '보네요',
'드뎌': '드디어',
'었어용': '었어요',
'딸래미랑': '딸이랑',
'마싯서용': '맛있어요',
'맛나고': '맛있고',
'맛있이요': '맛있어요',
'이인엽떡': '2인 엽떡',
'져여': '져요',
'매주먹어도': '매주 먹어도',
'햇네요': '했네요',
'맛두': '맛도',
'맵콤': '매콤',
'넓직하고': '넓적하고',
'매워여': '매워요',
'동대문엽기떡': '동대문엽기떡볶이',
'인데두': '인데도',
'맛나여': '맛나요',
'업떡': '엽떡',
'드시라요': '드세요',
'끗남': '끝남',
'해볼랍니다': '해보렵니다',
'배불러용': '배불러요',
'먹었어용': '먹었어요',
'는디': '는데',
'있네용': '있네요',
'있어융': '있어요',
'을라고': '으려고',
'았어용': '았어요',
'고냥': '그냥',
'먹습니당': '먹습니다',
'하나더': '하나 더',
'따로주셔서좋': '따로 주셔서 좋았어요',
'켯는데': '켰는데',
'왓어요': '왔어요',
'매장께': '매장 것이',
'엿던가': '였던가',
'마싯써요': '맛있어요',
'마싯': '맛있',
'엉망진찾': '엉망진창',
'기분잡치고': '기분 잡치고',
'먹눈': '먹는',
'씨씨 거리면서': '씩씩거리면서',
'먼가': '뭔가',
'머것어': '먹었어',
'햇슴니다': '했습니다',
'쫒아': '쫓아',
'앉쳐놓고': '앉혀놓고',
'본점치고': '본점 치고',
'움식': '음식',
'었슴다': '었습니다',
'리뉴얼': ' 리뉴얼 ',
'셀프로': ' 셀프로 ',
'맛응': '맛을',
'되버': '돼버',
'맛있어용': '맛있어요',
'매우 지난보다': '매우 전보다',
'짘짜': '진짜',
'말캉한떡': '말캉한 떡',
'낫어요': '났어요',
'앙이': '양이',
'마리로제': '마라로제',
'좋아요굳': '좋아요 굿',
'너무 지난보다': '너무 전보다',
'먿네요': '먹었네요',
'오쩔수': '어쩔수',
'갈려하니': '가려하니',
'들어갈려고': '들어가려고',
'있숩니다': '있습니다',
'져아': '좋아',
'소세지': '소시지',
'중당': '중국당면',
'중당면': '중국당면',
'베이칸': '베이컨',
'라구여': '라고요',
'멘날': '맨날',
'떡벆이': '떡볶이',
'프렌차이즈': '프랜차이즈',
'스트래스': '스트레스',
'유쾌하진': '유쾌하신',
'너어어무': '너무',
'너어무': '너무',
'기름찌고': '기름지고',
'습니당': '습니다',
'띰차는': '땀 차는',
'거에요': '거예요',
'조아하': '좋아하',
'요기로': '여기로',
'싶어용': '싶어요',
'하시구': '하시고',
'어무': '너무',
'맛잏어': '맛있어',
'네용': '네요',
'점바이점': '점바점',
'하다거': '하다고',
'욕싱': '욕심',
'이에여': '이에요',
'이예여': '이에요',
'오리닷': '오리다',
'냐구여': '냐고요',
'죠야요': '좋아요',
'마시쑴': '맛있음',
'오께요': '올게요',
'됬다': '됐다',
'임니다': '입니다',
'아여': '아요',
'렌지': '전자레인지',
'마싯서': '맛있어',
'먹엇': '먹었',
'되서': '돼서',
'네여': '네요',
'떡뽁이': '떡볶이',
'벌뻘': '벌벌',
'먹었지용': '먹었지요',
'먀장': '매장',
'멥찔': '맵찔이',
'이였어용': '이었어요',
'계란찌': '계란찜',
'려구': '려고',



    # – 여진 끝: 4259 – #

    # 추가 #

   '너무맛있어요': '너무 맛있어요',
    '직원분들친절하시공': '직원분들 친절하시고',
    '자주': '자주 ',
    '홀도좋고': '홀도 좋고',
    '양도많고': '양도 많고',
    '너무좋네요': '너무 좋네요',
    '처음먹어봤는대': '처음 먹어봤는데',
    '다돼서': '다 돼서',
    '마시써용': '맛있어요',
    '쟌맛입니당': '정말 맛있습니다',
    '다른곳 보다': '다른 곳 보다',
    '왜없나요': '왜 없나요?',
    '오리지날다': '오리지널 다',
    '타브렌드는' : '타브랜드는',
    '갠춘하네여' : '괜찮네요',
    '싱거워죽는줄알았어요': '싱거워 죽는 줄 알았어요',
    '듬뿍들어있어': '듬뿍 들어있어',
    '넓직하고' : '넓고',
    '아귀워요': '아쉬워요',
    '쥰맛입니다' : '정말 맛있습니다',
    '쿨픽스': '쿨피스',
    '마세요그냥': '마세요 그냥',
    '사각형모양이예요': '사각형 모양이에요',
    '위가부담': '위가 부담스러워요',
    '미쳐쏘요 짱맛있음' : '미쳤어요 정말 맛있음',
    '헤헹': '',
    '기분나쁜냄새남': '기분 나쁜 냄새 남',
    '하십니당': '하십니다',
    '하악': '',
    '있어으면합니다': '있었으면 합니다',
    '마라떡보키': '마라떡볶이',
    '메워요': '매워요',
    '잘맛고': '잘 맞고',
    '네가지없는게': '싸가지 없는 게',
    '않팔아서': '안 팔아서',
    '별루': '별로',
    '최고쵝': '최고',
    '너므': '너무',
    '맛있개': '맛있게',
    '세젤맛': '세상에서 제일 맛있어요',
    '욀케매운건지': '왜 이렇게 매운건지',
    '안조음': '안 좋음',
    '드셈요': '드세요',
    '추워용': '추워요',
    '워용': '워요',
    '조음': '좋음',
    '해주세요오': '해주세요',
    '엽대급으로': '역대급으로',
    '초큼': '조금',
    '냥냥해요': '낭낭해요',
    '굿이 영수증 달라고': '굳이 영수증 달라고',
    '묵었다': '먹었다',
    '먹었아요': '먹었어요',
    '성격개드럽고이딴대가는데 비위생적일거같다 인사성없고 영업되는게신기하네': '성격 개 더럽고 이딴 데 가는데 비위생적일 것 같다 인사성 없고 영업되는 게 신기하네',
    '멏번':'몇번',
    '붑상해서 제 안시키려고요': '기분 상해서 이제 안 시키려구요',


#윤영 님 파트#


'11 26 오후 5 55분 쯤':'11월 26일 오후 5시 55분쯤',
'2만원후반대로':'2만원 후반대로',
'2인엽떡':'2인 엽떡',
'4가지 없고':'싸가지 없고',
'ㄱㅊ':'괜찮',
'ㅂㄹ':'별로',
'ㅈㄴ':'정말',
'가섯비':'가성비',
'감사합니당':'감사합니다',
'감사합니당!':'감사합니다',
'감사합니당~':'감사합니다',
'같네염':'같네요',
'같네욤':'같네요',
'같네용':'같네요',
'같아여':'같아요',
'같아염':'같아요',
'같아요염':'같아요',
'같아요욤':'같아요',
'같아요용':'같아요',
'같아욤':'같아요',
'갠춘':'괜찮',
'계란찌':'계란찜',
'고맙습니당':'고맙습니다',
'괜차늠':'괜찮음',
'괜찬':'괜찮음',
'괜찬아요':'괜찮아요',
'괜찬았습니당':'괜찮았습니다',
'괜찬았어요':'괜찮았어요',
'괜찬음':'괜찮음',
'괜찮네용':'괜찮네요',
'괜찮습니당':'괜찮습니다',
'괜찮아용':'괜찮아요',
'괜찮았습니당':'괜찮았습니다',
'괜찮았어염':'괜찮았어요',
'괜찮았어요욤':'괜찮았어요',
'괜찮았어요용':'괜찮았어요',
'괜찮았어욤':'괜찮았어요',
'괜찮았어용':'괜찮았어요',
'괜찮은데염':'괜찮은데요',
'괜찮은데욤':'괜찮은데요',
'괜찮은데용':'괜찮은데요',
'굳':'굿',
'굳굳':'굿',
'굳굳굳':'좋아요',
'굳굳이에요':'좋아요',
'굳굳좋아요':'좋아요',
'굳이에요':'좋아요',
'굿':'좋아요',
'굿굿':'굿',
'굿굿 친절하세용':'굿굿 친절하세요',
'굿굿굿':'좋아요',
'굿굿굿굿':'좋아요',
'굿굿굿이에요':'좋아요',
'굿이에요':'좋아요',
'굿좋아요':'좋아요',
'귀찮아해서':'귀찮아서',
'그래도물좀넣고':'그래도 물 좀 넣고',
'그래슴':'그랬음',
'그랫슨':'그랬음',
'그럴수있어요':'그럴 수 있어요',
'그럴수있죠':'그럴 수 있죠',
'기달려서':'기다려서',
'기달렸어요':'기다렸어요',
'기달림':'기다림',
'기달림니다':'기다립니다',
'기분더러웠는데':'기분 더러웠는데',
'길가여서':'길가에 있어서',
'김밥은국룰이죠':'김밥은 국룰이죠',
'꺼같네요':'것 같네요',
'꾸르맛':'꿀맛',
'꾸르맛탱':'꿀맛이에요',
'꿀꿀할때':'꿀꿀할 때',
'꿀맛탱':'꿀맛이에요',
'날려지네여':'날려지네요',
'남알바':'남자 알바',
'너모':'너무',
'너모 맛있어요':'너무 맛있어요',
'너무 맛있습니당':'너무 맛있습니다',
'너무너누':'너무너무',
'너무너무':'너무',
'너무너무 더움':'너무너무 더워요',
'너무너무너무':'너무너무',
'너무너무너무너무':'너무너무',
'너무너무너무너무너무':'너무너무',
'너무너무맛있네요':'너무 맛있네요',
'너무너무맛있어요':'너무 맛있어요',
'너무맛있었어요':'너무 맛있었어요',
'너뮤':'너무',
'너어어어무':'너무',
'넘':'너무',
'넘 귀엽고':'너무 귀엽고',
'넘 맛':'너무 맛',
'넘 맛 있어여':'너무 맛있어요',
'넘넘':'너무너무',
'넘넘넘':'너무',
'넘넘넘넘':'너무',
'넘넘넘맛있어요':'너무 맛있어요',
'넘넘맛있어요':'너무 맛있어요',
'넘맛잇어요':'너무 맛있어요',
'넘맛있':'너무 맛있',
'넘맛있어요':'너무 맛있어요',
'넘조하욤':'너무 좋아요',
'넘좋아요':'너무 좋아요',
'넘좋았어요':'너무 좋았어요',
'넘흐넘흐':'너무',
'누무먼하':'너무 많아',
'느무':'너무',
'느므':'너무',
'늘 맛나요':'늘 맛있어요',
'다륻 지점에':'다른 지점에',
'다먹었넹':'다 먹었네요',
'단무지500원 으로바꾸는데':'단무지 500원으로 바꾸는데',
'닼':'다',
'대따':'아주',
'대따 잘 어울리고':'아주 잘 어울리고',
'대박이네용':'대박이네요',
'대박이에요':'정말 좋아요',
'대박이에용':'대박이에요',
'더 더보기':'nan',
'더러웠어요 셀프코너':'더러웠어요. 셀프 코너',
'더럽게할것같은':'더럽게 할 것 같은',
'더럽단':'더럽다는',
'더럽도':'더럽고',
'더워서 헥헥대면서도':'더워서 헉헉대면서도',
'덜매운맛':'덜 매운맛',
'덜매운맛시켰는데 오 매워요':'덜 매운 맛 시켰는데 오 매워요',
'도긴개긴':'도긴개긴이에요',
'돈 아깝네요':'돈이 아깝네요',
'되서':'돼서',
'되세용':'되세요',
'듈이':'둘이',
'드러워요':'더러워요',
'들어옴':'들어옵니다',
'떡복기':'떡볶이',
'떡복이':'떡볶이',
'떡복이는':'떡볶이는',
'떡볶기':'떡볶이',
'떡볶기 먹는날':'떡볶이 먹는 날',
'떡볶기가':'떡볶이가',
'떡볶이 떡 조합':'떡볶이와 떡의 조합',
'떡볶이국물':'떡볶이 국물',
'떡볶이는 엽기떢볶기가':'떡볶이는 엽기떡볶이가',
'떡볶이하나먹는데40분기다렸음':'떡볶이 하나 먹는데 40분 기다렸음',
'떢':'떡',
'떢볶기':'떡볶이',
'떢볶이':'떡볶이',
'로재':'로제',
'로제떡복기':'로제떡볶이',
'마딛':'맛있다',
'마라떡볶이은근':'마라떡볶이가 은근',
'마라로제떡볶이 처음 먹어보눈데':'마라로제떡볶이 처음 먹어보는데',
'마라로제로 유부 추가해서':'마라로제에 유부 추가해서',
'마라로제엽떡':'마라 로제 엽떡',
'마라로제엽떡이':'마라로제 엽떡이',
'마라맛 너무 맛있어요':'마라 맛 너무 맛있어요',
'마라엽떡 love':'마라엽떡 최고',
'마라엽떡 공덕점 최고입니당':'마라엽떡 공덕점 최고입니다',
'마라엽떡 기미':'마라엽떡 기대',
'마라엽떡 넘 맛있어요':'마라엽떡 너무 맛있어요',
'마라엽떡 중독입니당':'마라엽떡 중독입니다',
'마라엽떡 최고시당':'마라엽떡 최고입니다',
'마라엽떡너무맛있어요':'마라엽떡 너무 맛있어요',
'마라엽떡는':'마라엽떡은',
'마라엽떡최고시당':'마라엽떡 최고입니다',
'마라엽떡최고입니당':'마라엽떡 최고입니다',
'마세용':'마세요',
'마싯게':'맛있게',
'마싯고':'맛있고',
'마싯뇌요':'맛있네요',
'마싯다':'맛있다',
'마싯당':'맛있다',
'마싯슴다':'맛있습니다',
'마싯습니당':'맛있습니다',
'마싯어요':'맛있어요',
'마싯었러여여여':'맛있었어요',
'마싯었습니당':'맛있었습니다',
'마싯었어요':'맛있었어요',
'마싯엉':'맛있어요',
'마싰어요':'맛있어요',
'마포공덕점이 최고 맛나용':'마포공덕점이 최고 맛나요',
'맛 있어요':'맛있어요',
'맛나게':'맛있게',
'맛나네여':'맛있네요',
'맛나여':'맛나요',
'맛나요':'맛있어요',
'맛나욤':'맛나요',
'맛나용':'맛나요',
'맛남':'맛있음',
'맛도리':'맛돌이',
'맛앗어요':'맛있어요',
'맛이떠요':'맛있어요',
'맛이변했어요':'맛이 변했어요',
'맛잇':'맛있',
'맛잇게':'맛있게',
'맛잇게요':'맛있어요',
'맛잇네여':'맛있네요',
'맛잇네용':'맛있네요',
'맛잇는대':'맛있는데',
'맛잇는디':'맛있는데',
'맛잇당':'맛있다',
'맛잇당!':'맛있다',
'맛잇슴다':'맛있습니다',
'맛잇습니당':'맛있습니다',
'맛잇습니당당':'맛있습니다',
'맛잇어':'맛있어',
'맛잇어여':'맛있어요',
'맛잇어염':'맛있어요',
'맛잇어요':'맛있어요',
'맛잇어요오':'맛있어요',
'맛잇어요용':'맛있어요',
'맛잇어요용용':'맛있어요',
'맛잇어요용용용':'맛있어요',
'맛잇어욤':'맛있어요',
'맛잇어용':'맛있어요',
'맛잇어용오':'맛있어요',
'맛잇어용용':'맛있어요',
'맛잇어용용용':'맛있어요',
'맛잇엇습니당':'맛있었습니다',
'맛잇엇어여':'맛있었어요',
'맛잇엇어요':'맛있었어요',
'맛잇엇어용':'맛있었어요',
'맛잇었습니당':'맛있었습니다',
'맛잇었어염':'맛있었어요',
'맛잇었어요':'맛있었어요',
'맛잇었어욤':'맛있었어요',
'맛잇었어용':'맛있었어요',
'맛잇엉':'맛있어요',
'맛잇음':'맛있음',
'맛잇음당':'맛있어요',
'맛있 마라엽떡':'맛있는 마라엽떡',
'맛있게 잘먹었습니당':'맛있게 잘 먹었습니다',
'맛있게 잘먹었어요':'맛있게 잘 먹었어요',
'맛있게 잘목었습니다':'맛있게 잘 먹었습니다',
'맛있게 잘하세요':'맛있게 잘 하세요',
'맛있게묵엇습니더':'맛있게 먹었습니다',
'맛있게잘먹었습니다':'맛있게 잘 먹었습니다',
'맛있게잘먹었습니당':'맛있게 잘 먹었습니다',
'맛있게잘먹었습니당!':'맛있게 잘 먹었습니다',
'맛있게잘먹었어요':'맛있게 잘 먹었어요',
'맛있게잘목었어요':'맛있게 잘 먹었어요',
'맛있게잘묵었습니당':'맛있게 잘 먹었습니다',
'맛있게잘묵었어요':'맛있게 잘 먹었어요',
'맛있겠':'맛있게',
'맛있겠 잘 먹었습니다':'맛있게 잘 먹었습니다',
'맛있구':'맛있고',
'맛있구요':'맛있고요',
'맛있눈고야':'맛있는 거야',
'맛있당':'맛있다',
'맛있둠':'맛있음',
'맛있뜌':'맛있다',
'맛있습니당':'맛있습니다',
'맛있습니당!':'맛있습니다',
'맛있습니당~':'맛있습니다',
'맛있습니더':'맛있습니다',
'맛있습미당':'맛있습니다',
'맛있아사':'맛있어서',
'맛있어 요':'맛있어요',
'맛있어여':'맛있어요',
'맛있어염':'맛있어요',
'맛있어영':'맛있어요',
'맛있어오':'맛있어요',
'맛있어요 다만':'맛있어요. 다만',
'맛있어욤':'맛있어요',
'맛있어용':'맛있어요',
'맛있어욬':'맛있어요',
'맛있었러욪':'맛있었어요',
'맛있었습니당':'맛있었습니다',
'맛있었습니당당':'맛있었습니다',
'맛있었습미당':'맛있었습니다',
'맛있었어여':'맛있었어요',
'맛있었어염':'맛있었어요',
'맛있었어욤':'맛있었어요',
'맛있었어용':'맛있었어요',
'맛있었오요':'맛있어요',
'맛있었으나':'맛있었지만',
'맛있엉':'맛있어',
'맛있우요':'맛있어요',
'맛있잇어요':'맛있어요',
'맛잡':'맛집',
'맞아용':'맞아요',
'매번 잘 먹고 있어오':'매번 잘 먹고 있어요',
'매운 움식에':'매운 음식에',
'매워여':'매워요',
'매워염':'매워요',
'매워요 쵝오시다':'매워요 최고입니다',
'매워요요':'매워요',
'매워욤':'매워요',
'매워용':'매워요',
'매장에서 맛있게 잘먹었어요':'매장에서 맛있게 잘 먹었어요',
'매장왓늗데':'매장 왔는데',
'맵네용':'맵네요',
'맵다요':'맵네요',
'맵찔이':'매운 음식을 잘 못 먹는 사람',
'맵찔이라':'매운 음식을 잘 못 먹어서',
'맵찔이에겐':'매운 음식을 잘 못 먹는 사람에게는',
'먀장':'매장',
'머리카락 나왔어요 하':'머리카락이 나왔어요. 하',
'먹고싶다':'먹고 싶다',
'먹고싶어요':'먹고 싶어요',
'먹었습니당':'먹었습니다',
'먹었습니더':'먹었습니다',
'먹었어염':'먹었어요',
'먹었어욤':'먹었어요',
'먹었어용':'먹었어요',
'먹었오용':'먹었어요',
'먹었음돠':'먹었습니다',
'몬가':'뭔가',
'못먹엇어오':'못 먹었어요',
'무안하게':'민망하게',
'묵은쌀로':'묵은 쌀로',
'묵은쌀로 지은밥이었고 ':'묵은 쌀로 지은 밥이었고 ',
'바랬는데':'바랐는데',
'바삭치즈만두랑':'바삭치즈만두와',
'바삭치즈만두에':'바삭치즈만두를',
'받앗는데':'받았는데',
'받질않고':'받질 않고',
'받질않으시네요':'받질 않으시네요',
'배달로드세요':'배달로 드세요',
'배달비 사처넌':'배달비 사천원',
'배달시켜목는':'배달 시켜먹는',
'배달시켜서':'배달 시켜서',
'배달시켰는데':'배달 시켰는데',
'배달포장홀':'배달 포장 홀',
'배불러용':'배불러요',
'배불렀어염':'배불렀어요',
'배불렀어욤':'배불렀어요',
'배불렀어용':'배불렀어요',
'번창하세염':'번창하세요',
'번창하세욤':'번창하세요',
'번창하세용':'번창하세요',
'번창하세용!!':'번창하세요',
'번창하세용~':'번창하세요',
'보여주심':'보여주셨어요',
'보이길래':'보이기에',
'봤어염':'봤어요',
'봤어욤':'봤어요',
'봤어용':'봤어요',
'불손한태도로':'불손한 태도로',
'불어서':'불어서',
'불친절쓰':'불친절해요',
'불친절하세여':'불친절해요',
'불친절하십니다':'불친절합니다',
'불친절함이 치가떨릴정도':'불친절함이 치가 떨릴 정도',
'불친절해서':'불친절해서',
'불친절힙니다':'불친절합니다',
'뿔어서':'불어서',
'사랑입니당':'사랑입니다',
'사랑합니당':'사랑합니다',
'사랑해용':'사랑해요',
'사장님불러서':'사장님 불러서',
'서비스엉망이에요':'서비스 엉망이에요',
'설치하시는게어쩔지':'설치하시는 게 어떨지',
'스슥':'살짝',
'시캬서':'시켜서',
'싫엉':'싫어요',
'싸가지 없음':'불친절함',
'싸가지x':'싸가지 없어요',
'싸가지없음':'불친절함',
'써비스':'서비스',
'씹혀서':'씹혀서',
'아는척도':'아는 척도',
'아뭏든':'아무튼',
'아숩아숩':'아쉽아쉽',
'아쉬운거는':'아쉬운 것은',
'안닦았는지':'안 닦았는지',
'안되서':'안돼서',
'안좋은':'안 좋은',
'않좋아요':'안 좋아요',
'않좋은':'안 좋은',
'야옹야옹야옹야옹좀짜다야옹야옹':'좀 짜요',
'야징':'야지',
'얏 샤탸':'nan',
'양배추 파 등':'양배추, 파 등',
'어용':'어요',
'언재먹어도':'언제 먹어도',
'없슴':'없음',
'없어용':'없어요',
'에어컨좀':'에어컨 좀',
'여성알바생':'여자 알바생',
'여알바분':'여자 알바분',
'역쉬':'역시',
'역시엽떡':'역시 엽떡',
'엽기 떡볶이 를':'엽기떡볶이를',
'엽기닭도리탕':'엽기 닭도리탕',
'엽기떡보끼':'엽기떡볶이',
'엽기떢볶기':'엽기떡볶이',
'엽떠':'엽떡',
'엽떡 여의도에서 먹을수 있는 자리가 있어요':'엽떡 여의도에서 먹을 수 있는 자리가 있어요',
'엽떡 입니데':'엽떡입니다',
'엽떡 입니데요':'엽떡입니다',
'엽떡 입니데이':'엽떡입니다',
'엽떡 존맛탱쓰':'엽떡 정말 맛있어요',
'엽떡너무맛있습니당':'엽떡 너무 맛있습니다',
'엽떡너무맛있습니당!':'엽떡 너무 맛있습니다',
'엽떡너무맛있어요':'엽떡 너무 맛있어요',
'엽떡너무맛있어용':'엽떡 너무 맛있어요',
'엽떡너무좋아여':'엽떡 너무 좋아요',
'엽떡너무좋아요':'엽떡 너무 좋아요',
'엽떡너무좋아용':'엽떡 너무 좋아요',
'엽떡마싯다':'엽떡 맛있다',
'엽떡맛잇어요':'엽떡 맛있어요',
'엽떡맛있습니당':'엽떡 맛있습니다',
'엽떡맛있어용':'엽떡 맛있어요',
'엽떡머는날':'엽떡 먹는 날',
'엽떡앱':'엽떡 앱',
'엽떡앱에는':'엽떡 앱에는',
'엽떡오리지널인데':'엽떡 오리지널인데',
'엽떡이아니라무슨':'엽떡이 아니라 무슨',
'엽떡이쥬':'엽떡이죠',
'엽똑':'엽떡',
'예쁘게 못올린걸말한거':'예쁘게 못 올린 걸 말한 거',
'예용':'이에요',
'오눌따라':'오늘따라',
'오늘은유독짠듯해요':'오늘은 유독 짠 듯해요',
'오리지날':'오리지널',
'오옹':'오',
'오홍':'오',
'오홍홍':'오',
'완젼':'완전',
'왕맛탱':'왕맛',
'욤':'요',
'욥':'요',
'욬':'요',
'웠구':'웠고',
'을매나':'얼마나',
'읭했는데':'의아했는데',
'이걸 먹으라고요':'이걸 먹으라는 건가요',
'이죵':'이죠',
'이쥬':'이죠',
'일한지 3일차인데':'일한 지 3일 차인데',
'입니당':'입니다',
'입니당!':'입니다',
'입니당^^':'입니다',
'입니당~':'입니다',
'입니당요':'입니다',
'입니데이':'입니다',
'잇습니당':'있습니다',
'잇어요':'있어요',
'잇엇을거에요':'있었을 거예요',
'잇었어요':'있었어요',
'있슴':'있음',
'있습니당':'있습니다',
'있습니데이':'있습니다',
'있어용':'있어요',
'자시':'잠시',
'잘먹엇습니당':'잘 먹었습니다',
'잘먹엇어요':'잘 먹었어요',
'잘먹었습니다':'잘 먹었습니다',
'잘먹었습니당':'잘 먹었습니다',
'잘먹었어요':'잘 먹었어요',
'잘먹었어요용':'잘 먹었어요',
'잘먹었어욤':'잘 먹었어요',
'잘먹었어용':'잘 먹었어요',
'잘묵엇숩니더':'잘 먹었습니다',
'잘묵엇습니당':'잘 먹었습니다',
'잘묵엇습니더':'잘 먹었습니다',
'잘묵엇어요':'잘 먹었어요',
'잘묵었습니당':'잘 먹었습니다',
'잘묵었어요':'잘 먹었어요',
'잘묵었어용':'잘 먹었어요',
'전에안그랬는데독특한향기가나네요':'전에 안 그랬는데 독특한 향기가 나네요',
'젤':'제일',
'젤 맛있는 곳':'제일 맛있는 곳',
'젤루':'제일',
'젤루 짜요':'제일 짜요',
'졍말':'정말',
'조아요':'좋아요',
'조하요':'좋아요',
'조하용':'좋아요',
'존맛탱':'존맛',
'존맛탱이에요':'정말 맛있어요',
'졸라':'nan',
'좋공':'좋고',
'좋네여':'좋네요',
'좋습니당':'좋습니다',
'좋습니댜':'좋습니다',
'좋아여':'좋아요',
'좋아염':'좋아요',
'좋아오':'좋아요',
'좋아요요':'좋아요',
'좋아요용':'좋아요',
'좋아욤':'좋아요',
'좋아용':'좋아요',
'좋아용용':'좋아요',
'좋아용용용':'좋아요',
'좋아하는 엽떡에':'좋아하는 엽떡은',
'좋앗슴다':'좋았습니다',
'좋앗습니다':'좋았습니다',
'좋앗습니다용':'좋았습니다',
'좋앗습니당':'좋았습니다',
'좋앗습니당~':'좋았습니다',
'좋앗어염':'좋았어요',
'좋앗어영':'좋았어요',
'좋앗어요':'좋았어요',
'좋앗어요용':'좋았어요',
'좋앗어욤':'좋았어요',
'좋앗어용':'좋았어요',
'좋았습니당':'좋았습니다',
'좋았습니더':'좋았습니다',
'좋았습미당':'좋았습니다',
'좋았어염':'좋았어요',
'좋았어요':'좋았어요',
'좋았어요염':'좋았어요',
'좋았어요욤':'좋았어요',
'좋았어요용':'좋았어요',
'좋았어욤':'좋았어요',
'좋았어용':'좋았어요',
'좋앙':'좋아요',
'좋앜':'좋았어요',
'좋을것 같아요':'좋을 것 같아요',
'좋이않았어요':'좋지 않았어요',
'좋타':'좋다',
'죠아요':'좋아요',
'죤맛':'존맛',
'주먹 김밥 에':'주먹김밥에',
'주먹김밥 완전 질어서 떡밥이고':'주먹김밥이 완전 질어서 떡밥 같고',
'주문누락이라는데 말이라도미리해주지 계속기다리라는건 무슨소리임 다시는여기안올듯':'주문 누락이라는데 말이라도 미리 해주지, 계속 기다리라는 건 무슨 소리입니까. 다시는 여기 안 올 듯.',
'주뮨':'주문',
'쥔장':'주인장',
'지역여행중일때':'지역 여행 중일 때',
'지은밥이었고':'지은 밥이었고',
'지저분한적이':'지저분한 적이',
'지저분해요숟가락에는':'지저분해요. 숟가락에는',
'직원부이':'직원분이',
'진심개별로임':'진짜 별로예요',
'진짜진짜':'정말',
'진철합니다':'친절합니다',
'징짜':'진짜',
'짜용':'짜요',
'짧은 남자직원교육좀':'짧은 남자 직원 교육 좀',
'짯네용':'짰네요',
'짯던거':'짰던 것',
'짯던거같아요':'짰던 것 같아요',
'짯어염':'짰어요',
'짯어요':'짰어요',
'짯어욤':'짰어요',
'짯어용':'짰어요',
'짱맛':'정말 맛있어요',
'짱맛탱이에요':'정말 맛있어요',
'짱이에용':'짱이에요',
'짱짱':'짱',
'쫀맛':'정말 맛있어요',
'착항맛':'착한맛',
'초보 맛있었요':'초보 맛있었어요',
'최고에여':'최고예요',
'최고에용':'최고예요',
'최고예요':'최고예요',
'최고입니당':'최고입니다',
'최고최고':'최고',
'최공':'최고',
'최구':'최고',
'최악이었습니당':'최악이었습니다',
'쵝오':'최고',
'쵝오에요':'최고예요',
'쵝오입니당':'최고입니다',
'치즈올리는거':'치즈 올리는 거',
'친절쓰':'친절해요',
'친절하세여':'친절해요',
'칭구들':'친구들',
'테이블고 닦지':'테이블도 닦지',
'퍽 불친절해요':'매우 불친절해요',
'포장주문 이용했어요':'포장 주문 이용했어요',
'포장주문하는데':'포장 주문하는데',
'퐁당치즈민두':'퐁당치즈만두',
'피한다고다가아니잖아요':'피한다고 다가 아니잖아요',
'하세여':'하세요',
'하세염':'하세요',
'하세욤':'하세요',
'하세용':'하세요',
'하시공':'하시고',
'합니다용':'합니다',
'해따':'했다',
'해용':'해요',
'핸드폰하기':'휴대폰 하기',
'했네용':'했네요',
'했습니다용':'했습니다',
'했습니당':'했습니다',
'했습니더':'했습니다',
'했습니데':'했습니다',
'했습니데이':'했습니다',
'했습니디':'했습니다',
'했어염':'했어요',
'했어욤':'했어요',
'했어용':'했어요',
'향신료때료부으셧나요':'향신료 때려 부으셨나요',
'허엉어어어또먹구싶다아':'또 먹고 싶다',
'험악해지는거':'험악해지는 거',
'헹주로':'행주로',
'힙니다':'합니다',
'가기':'가기에',
'가는곳':'가는 곳',
'가세용':'가세요',
'가져다 달라고 그러니까':'가져다 달라고 하니까',
'가태요':'같아요',
'강동구청점맛':'강동구청점 맛',
'개많아요':'아주 많아요',
'굿':'좋아요',
'기미':'기대',
'꿀맛탱':'정말 맛있어요',
'끓여먹을':'끓여 먹을',
'나올때까지':'나올 때까지',
'냄새안나고':'냄새 안 나고',
'넘 ':'너무 ',
'놀라염':'놀라요',
'누락되기때문에':'누락되기 때문에',
'느끼할줄':'느끼할 줄',
'늦게까지 해서 매장서':'늦게까지 영업해서 매장에서',
'돈아까워서':'돈 아까워서',
'들어주십니다':'들어주지 않습니다',
'땡길때':'땡길 때',
'랩퍼':'래퍼',
'마감전에주문했다고':'마감 전에 주문했다고',
'막나게':'마음껏',
'많은것':'많은 것',
'맛나서':'맛있어서',
'맛나용':'맛나요',
'맛을 맛을':'맛을',
'맛이어요':'맛있어요',
'맛있네용':'맛있네요',
'맛있뗭':'맛있어요',
'맛있을거':'맛있을 거',
'맞는듯':'맞는 듯',
'맞을수도':'맞을 수도',
'맞춰오네요':'맞춰 오네요',
'맵찌리':'매운 음식을 잘 못 먹는 사람',
'먹고싶은날':'먹고 싶은 날',
'먹고싶을까요':'먹고 싶을까요',
'먹고싶을때':'먹고 싶을 때',
'먹어보다니':'먹어 보다니',
'먹엇':'먹었',
'먹엇슴다':'먹었습니다',
'먹었습니당':'먹었습니다',
'메운맛':'매운맛',
'몇번을':'몇 번을',
'목참지':'못 참지',
'무슨일이있어도':'무슨 일이 있어도',
'문열어봤더니':'문 열어봤더니',
'바꼈네요':'바뀌었네요',
'변치않고':'변치 않고',
'불은거':'불은 거',
'사랑이쥬':'사랑이죠',
'사먹어봤습니다':'사 먹어봤습니다',
'사먹어요':'사 먹어요',
'살고싶다':'살고 싶다',
'세여':'세요',
'소세지':'소시지',
'시켜먹습니다':'시켜 먹습니다',
'시켜먹어봤는데':'시켜 먹어봤는데',
'싸오긴':'싸 오긴',
'씹힐때':'씹힐 때',
'안가져다':'안 가져다',
'안느껴져요':'안 느껴져요',
'안드네요':'안 드네요',
'안떠서':'안 떠서',
'안와서':'안 와서',
'안좋아하는데':'안 좋아하는데',
'알바':'알바생',
'앱에서 포장3000원 할인':'앱에서 포장 3,000원 할인',
'양도많고':'양도 많고',
'양많고':'양 많고',
'양이많고':'양이 많고',
'없슴':'없음',
'여기때문에':'여기 때문에',
'엽덕':'엽떡',
'엽떡는':'엽떡은',
'엽오는':'엽떡은',
'옆떡':'엽떡',
'오뎅':'어묵',
'오래걸리기':'오래 걸리기',
'오랫만에':'오랜만에',
'요청사항도':'요청 사항도',
'이번에능':'이번에는',
'있어용':'있어요',
'자리로 가져다 주셔요':'자리로 가져다 주세요',
'적을테니':'적을 테니',
'전화주문':'전화 주문',
'접어라':'접으세요',
'조아용':'좋아요',
'주문가능해서':'주문 가능해서',
'중독되는맛입니다':'중독되는 맛입니다',
'중화시키고 어우러주니':'중화시켜 어우러지니',
'즐겨먹는편임':'즐겨 먹는 편임',
'쪼금':'조금',
'찾으러가면':'찾으러 가면',
'쳐갔네':'치고 갔네요',
'추천드립니다':'추천합니다',
'출시 됬다':'출시됐다',
'치즈추가':'치즈 추가',
'치즈추가 소시지 추가':'치즈 추가, 소시지 추가',
'터진건':'터진 건',
'평타이상':'평타 이상',
'필요한듯':'필요한 듯',
'해먹으니':'해 먹으니',
'해줘요':'해 줘요',
'홀일때':'홀일 때',
'넘':'너무',
'넘 맛':'너무 맛',
'넘좋아요':'너무 좋아요',
'맛나요':'맛있어요',
'번창하세용':'번창하세요',

    # 추출 실수

    '더보기': '...'
}

# 교정 함수
def refine_text(text):
    if pd.isna(text):
        return text
    refined = str(text)
    for wrong, correct in replace_dict.items():
        refined = refined.replace(wrong, correct)
    return refined

# Refined_Text 열 생성
review['Refined_Text'] = review['Cleaned_Text'].apply(refine_text)

# 결과 확인
display(review[['Cleaned_Text', 'Refined_Text']].head(1000))

# CSV로 저장 (필요 시)
review.to_csv("refined_reviews.csv", index=False, encoding='utf-8-sig')
print("✅ Refined_Text 생성 및 저장 완료! refined_reviews.csv 파일로 저장되었습니다.")

,Cleaned_Text,Refined_Text
0,로제반반 전화로 주문하여 방문으로 가져간 사람입니다 너무 맛있게 잘먹었습니다 신선한...,로제반반 전화로 주문하여 방문으로 가져간 사람입니다 너무 맛있게 잘 먹었습니다 신선...
1,마라엽떡 맛있어요 밥 비벼먹어도 굿,마라엽떡 맛있어요 밥 비벼먹어도 좋아요
2,너무맛있어요 직원분들친절하시공 자주애용해요,너무 맛있어요 직원분들친절하시고 자주 애용해요
3,2층 홀도좋고 친절하시고 양도많고 너무좋네요,2층 홀도 좋고 친절하시고 양도 많고 너무 좋네요
4,엽닭 존맛입니다,엽기닭발 존맛입니다
...,...,...
995,엽기 착한맛,엽기 착한맛
996,맛있음 주인 바뀌어서 친절해짐,맛있음 주인 바뀌어서 친절해짐
997,난 응암점 맛있는디 근데 직원들 친절한지는 모르겠슴,난 응암점 맛있는데 근데 직원들 친절한지는 모르겠슴
998,여기 사장님 바뀌시고 괜찮아져서 자주 시켜먹다가 픽업으로 한번 찾으러갔는데 알바 싹...,여기 사장님 바뀌시고 괜찮아져서 자주 시켜먹다가 픽업으로 한번 찾으러갔는데 알바생...


✅ Refined_Text 생성 및 저장 완료! refined_reviews.csv 파일로 저장되었습니다.


In [ ]:
from google.colab import files
import pandas as pd
import os

# 데이터프레임이 로드되어 있는지 확인
if 'review' not in locals():
    print("오류: 'review' 데이터프레임을 찾을 수 없습니다. 데이터를 먼저 로드해주세요.")
else:
    # 현재 열 목록 가져오기
    current_columns = review.columns.tolist()

    # 이동 및 삭제할 열 이름 정의
    column_to_move = 'Refined_Text'
    column_to_drop = 'Tokenized_Text'
    target_column = 'Cleaned_Text'

    # 이동할 열과 삭제할 열을 현재 목록에서 제거
    if column_to_move in current_columns:
        current_columns.remove(column_to_move)
    if column_to_drop in current_columns:
        current_columns.remove(column_to_drop)

    # target_column의 인덱스 찾기
    try:
        target_index = current_columns.index(target_column)
        # target_column의 오른쪽 (target_index + 1)에 column_to_move 삽입
        current_columns.insert(target_index + 1, column_to_move)

        # 새로운 순서로 데이터프레임 열 재배열
        review_modified = review[current_columns].copy() # 수정된 데이터프레임 복사

        print(f"✅ '{column_to_move}' 열을 '{target_column}' 열 오른쪽으로 이동하고 '{column_to_drop}' 열을 삭제 완료.")
        display(review_modified.head())

        # 수정된 데이터프레임을 CSV 파일로 저장
        modified_file_name = 'refined_reviews.csv' # 사용자 요청에 따라 기존 이름으로 저장
        # Google Drive 경로에 저장
        drive_path_for_save = '/content/drive/MyDrive/Colab Notebooks/' + modified_file_name
        review_modified.to_csv(drive_path_for_save, index=False, encoding='utf-8-sig')
        print(f"✅ 수정된 데이터프레임을 구글 드라이브 '{drive_path_for_save}' 파일로 저장 완료.")

        # 저장된 파일 다운로드 (구글 드라이브 경로에서 다운로드 시도)
        try:
            from google.colab import drive
            drive.mount('/content/drive') # 마운트 확인
            files.download(drive_path_for_save)
            print(f"✅ '{drive_file_path}' 파일 다운로드 링크가 생성되었습니다.") # Note: drive_file_path is not defined here, should use drive_path_for_save
        except FileNotFoundError:
            print(f"오류: 구글 드라이브에서 '{drive_path_for_save}' 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
        except Exception as e:
            print(f"파일 다운로드 중 오류 발생: {e}")


    except ValueError as e:
        print(f"오류: 필요한 열이 데이터프레임에 없습니다. {e}")
        print("현재 데이터프레임 열:", review.columns.tolist())

✅ 'Refined_Text' 열을 'Cleaned_Text' 열 오른쪽으로 이동하고 'Tokenized_Text' 열을 삭제 완료.


,Review_UID,Raw_Text,Cleaned_Text,Refined_Text
0,A1_S01_R001,로제반반 전화로 주문하여 방문으로 가져간 사람입니다! 너무 맛있게 잘먹었습니다!! ...,로제반반 전화로 주문하여 방문으로 가져간 사람입니다 너무 맛있게 잘먹었습니다 신선한...,로제반반 전화로 주문하여 방문으로 가져간 사람입니다 너무 맛있게 잘 먹었습니다 신선...
1,A1_S01_R002,마라엽떡 맛있어요. 밥 비벼먹어도 굿!,마라엽떡 맛있어요 밥 비벼먹어도 굿,마라엽떡 맛있어요 밥 비벼먹어도 좋아요
2,A1_S01_R003,너무맛있어요~~~~!!!!!! 직원분들친절하시공ㅎ 자주애용해요ㅎ,너무맛있어요 직원분들친절하시공 자주애용해요,너무 맛있어요 직원분들친절하시고 자주 애용해요
3,A1_S01_R004,2층 홀도좋고 친절하시고 양도많고 너무좋네요~~,2층 홀도좋고 친절하시고 양도많고 너무좋네요,2층 홀도 좋고 친절하시고 양도 많고 너무 좋네요
4,A1_S01_R006,엽닭 존맛입니다,엽닭 존맛입니다,엽기닭발 존맛입니다


✅ 수정된 데이터프레임을 구글 드라이브 '/content/drive/MyDrive/Colab Notebooks/refined_reviews.csv' 파일로 저장 완료.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

파일 다운로드 중 오류 발생: name 'drive_file_path' is not defined
